# 01 · Statistical tests

Notebook net per preparar dades, definir outcomes i calcular Kruskal, Spearman, pairwise adjacent i Chi-square.

In [ ]:
from pathlib import Path
import pandas as pd
from create_groups_comf import add_outcomes_grup
from create_groups_sens import add_outcomes_group_sensation
from utils_stats import *


PROJECT_ROOT = Path.cwd().resolve().parent
DATA_DIR = Path("../../data")

votes = pd.read_csv(DATA_DIR / "all_surveys(votes).csv")
stops = pd.read_csv(DATA_DIR / "all_surveys(stops).csv")
ids = pd.read_csv(DATA_DIR / "all_surveys(ID).csv")

df = votes.merge(stops, on="space_code", how="left", suffixes=("", "_stop"))
print(df.shape)
df.head()
#df = add_outcomes_grup(df)
df = add_outcomes_group_sensation(df)


In [ ]:
NUMERIC_PREDICTORS = [
    "<T-T_fixed+<T>>","<HDX-HDX_fixed+<HDX>>","SVF_1.5m","NDVI_1.5m","climate_shelter",
    "distance_green_zone","distance_drinking_fountain","distance_fountain","IVAC"
]
CATEGORICAL_PREDICTORS = [
    "sun","wind","thermal_comfort_walking","surface_type","space_category","space_category2",
    "bad_smell","noise_level","people_within_10m_radius","age2","gender","participant_clothing"
]
OUTCOME_ORDERS = {
    "comfort3": ["comfortable","neutral","uncomfortable"],
    "comfort3_option1": ["comfortable-neutral","uncomfortable","very uncomfortable"],
    "comfort3_option2": ["comfortable", "neutral-uncomfortable","very uncomfortable"],
    "comfort3_UNoption": ["not_uncomfortable", "uncomfortable","very uncomfortable"],  
    "comfort4_soft": ["comfortable","near_neutral","slightly_uncomfortable","uncomfortable"],
    "comfort4_option1":["very comfortable","near_neutral","slightly_uncomfortale","uncomfortable"],
    "comfort4_option2":["very comfortable","near_neutral","uncomfortale","very uncomfortable"]
}
OUTCOME_ORDERS_SENSATION = {
    "sens7": ["cool","slightly cool","neutral","slightly warm","warm","hot","very hot"],
    "sens3": ["cool","neutral","warm"],
    "sens3_option1": ["cool","neutral-warm","hot"],
    "sens3_option2": ["cool-neutral","warm-hot","very hot"],
    "sens3_option3": ["cool","neutral-slightly warm","warm-hot"],
    "sens4_option1": ["cool","slightly-neutral","warm-hot","very hot"],
    "sens4_option2": ["cool","neutral","warm-hot","very hot"]
}

In [ ]:
from tqdm.auto import tqdm
kw_results, sp_results, pw_results = [], [], []
for outcome_col, order in tqdm(OUTCOME_ORDERS_SENSATION.items()):
    for predictor_col in NUMERIC_PREDICTORS:
        if predictor_col not in df.columns: 
            continue
        kw = kruskal_test(df, outcome_col, predictor_col)
        if kw is not None: kw_results.append(kw)
        sp = spearman_monotonicity(df, outcome_col, predictor_col, order)
        if sp is not None: sp_results.append(sp)
        pw = pairwise_adjacent_mwu(df, outcome_col, predictor_col, order)
        if not pw.empty: pw_results.append(pw)

kw_df = pd.DataFrame(kw_results).sort_values(["outcome","p_value","H"])
sp_df = pd.DataFrame(sp_results).sort_values(["outcome","p_value","spearman_rho"], ascending=[True,True,False])
pw_df = pd.concat(pw_results, ignore_index=True)
kw_df.head(), sp_df.head()
pw_df.head(100)

In [ ]:
chi_results = []
for outcome_col in OUTCOME_ORDERS_SENSATION.keys():
    for predictor_col in CATEGORICAL_PREDICTORS:
        if predictor_col not in df.columns:
            continue
        out = chisq_test(df, outcome_col, predictor_col)
        if out is not None:
            chi_results.append(out)
chi_df = pd.DataFrame(chi_results).sort_values(["outcome","p_value","cramers_v"], ascending=[True,True,False])
chi_df.head(60)

In [ ]:
kw_df.to_csv("csvs_sens/cat_kruskal_wallis_results.csv", index=False)
sp_df.to_csv("csvs_sens/cat_sp_df.csv", index=False)
pw_df.to_csv("csvs_sens/cat_pairwise_adjacent_mannwhitney_results.csv", index=False)
chi_df.to_csv("csvs_sens/cat_chi_square_cramersv_results.csv", index=False)